# Notebook 11 — Agentes de Planificación y Sistemas Multi-Agente

**Semana:** 11  
**Bloque:** 5.3 — Orquestación avanzada  
**Tiempo estimado:** 4-5 horas

## Qué vas a construir

- Un planificador que descompone una tarea compleja en subtareas estructuradas
- Un ejecutor que corre el plan paso a paso con replanificación ante fallos
- Un sistema multi-agente Supervisor + Workers para análisis de dominio
- **Reto final:** pipeline de inteligencia competitiva que combina investigación, análisis y redacción

## Prerrequisitos

Semanas 9 y 10 completadas. Herramientas del notebook 09 disponibles.

```bash
pip install openai requests python-dotenv
```

---
## Celda 0 — Setup y herramientas base

In [ ]:
import os
import json
import math
import uuid
import time
import datetime
import requests
from dataclasses import dataclass, field, asdict
from typing import Callable, Any
from enum import Enum
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
cliente = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODELO  = "gpt-4o-mini"

# ── Helpers LLM ──────────────────────────────────────────────────────────────

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.2,
    max_tokens: int = 1024
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(mensajes: list[dict], system: str | None = None) -> dict:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs, temperature=0.0,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

# ── Herramientas (del notebook 09) ───────────────────────────────────────────

def buscar_wikipedia(query: str, sentences: int = 4) -> str:
    url    = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "format": "json", "list": "search",
               "srsearch": query, "srlimit": 1}
    try:
        res    = requests.get(url, params=params, timeout=8).json()
        if not res["query"]["search"]:
            return f"No results for '{query}'."
        titulo = res["query"]["search"][0]["title"]
        p2     = {"action": "query", "format": "json", "titles": titulo,
                  "prop": "extracts", "exsentences": sentences,
                  "exintro": True, "explaintext": True}
        pag    = requests.get(url, params=p2, timeout=8).json()
        texto  = list(pag["query"]["pages"].values())[0].get("extract", "")
        return f"[Wikipedia: {titulo}]\n{texto}"
    except Exception as e:
        return f"Wikipedia error: {e}"

def calcular(expresion: str) -> str:
    permitidos = {"__builtins__": {}, "abs": abs, "round": round,
                  "sqrt": math.sqrt, "log": math.log, "pi": math.pi}
    try:
        return f"{expresion} = {eval(expresion, permitidos)}"
    except Exception as e:
        return f"Calc error: {e}"

def obtener_fecha() -> str:
    return datetime.datetime.utcnow().strftime("%Y-%m-%d")

# Registro de herramientas disponibles para los agentes
HERRAMIENTAS: dict[str, Callable] = {
    "buscar_wikipedia": buscar_wikipedia,
    "calcular":         calcular,
    "obtener_fecha":    lambda: obtener_fecha(),
}

print("Setup completado.")
print(f"Herramientas disponibles: {list(HERRAMIENTAS.keys())}")

---
## Celda 1 — Planificador: descompone una tarea en subtareas

El planificador recibe una tarea compleja y genera un plan estructurado.
El plan incluye dependencias entre pasos para detectar paralelización.

In [ ]:
class EstadoPaso(str, Enum):
    PENDIENTE  = "pendiente"
    EN_CURSO   = "en_curso"
    COMPLETADO = "completado"
    FALLIDO    = "fallido"
    SALTADO    = "saltado"


@dataclass
class Subtarea:
    id:           str
    descripcion:  str
    herramienta:  str | None        # nombre de la herramienta, o None si es LLM directo
    argumentos:   dict | None       # argumentos de la herramienta
    depende_de:   list[str]         # IDs de pasos que deben completarse antes
    resultado:    str | None = None
    error:        str | None = None
    estado:       EstadoPaso = EstadoPaso.PENDIENTE


@dataclass
class Plan:
    id:       str
    objetivo: str
    pasos:    list[Subtarea]
    estado:   str = "borrador"      # "borrador" | "en_ejecucion" | "completado"

    def pasos_listos(self) -> list[Subtarea]:
        """Pasos cuyas dependencias ya están completadas."""
        completados = {p.id for p in self.pasos if p.estado == EstadoPaso.COMPLETADO}
        return [
            p for p in self.pasos
            if p.estado == EstadoPaso.PENDIENTE
            and all(dep in completados for dep in p.depende_de)
        ]

    def esta_completo(self) -> bool:
        return all(p.estado in (EstadoPaso.COMPLETADO, EstadoPaso.SALTADO) for p in self.pasos)

    def mostrar(self) -> None:
        iconos = {
            EstadoPaso.PENDIENTE:  "○",
            EstadoPaso.EN_CURSO:   "◎",
            EstadoPaso.COMPLETADO: "●",
            EstadoPaso.FALLIDO:    "✗",
            EstadoPaso.SALTADO:    "—",
        }
        print(f"\nPlan: {self.id} | Estado: {self.estado}")
        print(f"Objetivo: {self.objetivo}")
        print("-" * 55)
        for p in self.pasos:
            icono = iconos.get(p.estado, "?")
            herr  = f"[{p.herramienta}]" if p.herramienta else "[LLM]"
            deps  = f" ← {p.depende_de}" if p.depende_de else ""
            print(f"  {icono} {p.id}: {p.descripcion[:50]}{deps}")
            if p.resultado:
                print(f"      → {p.resultado[:70]}...")


SYSTEM_PLANIFICADOR = """You are a task planner for an AI agent.
Given a complex objective, decompose it into concrete, executable steps.
Each step should either:
  - Call one specific tool (buscar_wikipedia, calcular, obtener_fecha)
  - Or be a synthesis step (herramienta: null) done directly by the LLM

Rules:
- Maximum 6 steps per plan
- Each step must be independently executable
- Mark dependencies accurately: step B depends on A only if B NEEDS A's result
- Always end with a synthesis step that integrates all findings"""


def crear_plan(objetivo: str) -> Plan:
    """
    Genera un plan estructurado para el objetivo dado.
    El LLM decide cuántos pasos y qué herramientas usar.
    """
    herramientas_desc = json.dumps([
        {"nombre": "buscar_wikipedia", "para": "factual background information",
         "argumentos": {"query": "string", "sentences": "int (1-6)"}},
        {"nombre": "calcular",         "para": "mathematical expressions",
         "argumentos": {"expresion": "valid Python math expression"}},
        {"nombre": "obtener_fecha",    "para": "current date",
         "argumentos": {}},
    ])

    prompt = f"""Create a step-by-step plan to accomplish this objective:
OBJECTIVE: {objetivo}

Available tools:
{herramientas_desc}

Return JSON:
{{
  "pasos": [
    {{
      "id": "step_1",
      "descripcion": "What this step does",
      "herramienta": "buscar_wikipedia" | "calcular" | "obtener_fecha" | null,
      "argumentos": {{...}} | null,
      "depende_de": []
    }}
  ]
}}"""

    datos = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_PLANIFICADOR)

    pasos = [
        Subtarea(
            id=p["id"],
            descripcion=p["descripcion"],
            herramienta=p.get("herramienta"),
            argumentos=p.get("argumentos") or {},
            depende_de=p.get("depende_de", [])
        )
        for p in datos["pasos"]
    ]

    return Plan(
        id=f"plan_{uuid.uuid4().hex[:6]}",
        objetivo=objetivo,
        pasos=pasos
    )


# ── Genera y muestra planes para distintos objetivos ──────────────────────────
OBJETIVOS = [
    "Compare PostgreSQL and MongoDB: history, use cases, and key differences.",
    "Calculate how many days ago Python was first released (February 20, 1991) and summarize its history.",
]

planes_generados = []
for obj in OBJETIVOS:
    plan = crear_plan(obj)
    plan.mostrar()
    planes_generados.append(plan)

    # Muestra qué pasos pueden ejecutarse en paralelo
    listos = plan.pasos_listos()
    print(f"\n  Pasos ejecutables ahora ({len(listos)}): {[p.id for p in listos]}")

---
## Celda 2 — Ejecutor: corre el plan con replanificación

El ejecutor recorre el plan paso a paso.
Si un paso falla, intenta replanificar el resto en lugar de abortar.

In [ ]:
SYSTEM_REPLANIFICADOR = """You are replanning an AI agent's steps after a failure.
Given the failed step and the remaining plan, suggest how to proceed.
Be practical: sometimes the best option is to skip and continue with available information."""

SYSTEM_SINTETIZADOR = """You are synthesizing the results of a multi-step research plan.
Integrate all findings into a coherent, well-structured response.
Be factual and cite which step provided each piece of information.
If some steps failed or were skipped, note what information is missing."""


def ejecutar_paso(paso: Subtarea, contexto: dict[str, str]) -> str:
    """
    Ejecuta un único paso del plan.
    Si tiene herramienta: la llama.
    Si no: usa el LLM con los resultados anteriores como contexto.
    """
    if paso.herramienta:
        if paso.herramienta not in HERRAMIENTAS:
            return f"Error: herramienta '{paso.herramienta}' no disponible."
        argumentos = paso.argumentos or {}
        # Sustituye referencias a resultados anteriores en los argumentos
        argumentos_resueltos = {
            k: v.format(**contexto) if isinstance(v, str) else v
            for k, v in argumentos.items()
        }
        try:
            return HERRAMIENTAS[paso.herramienta](**argumentos_resueltos)
        except TypeError:
            return HERRAMIENTAS[paso.herramienta](**argumentos)
    else:
        # Paso de síntesis: el LLM integra los resultados anteriores
        contexto_str = "\n".join(
            f"[{id_paso}] {resultado[:300]}" for id_paso, resultado in contexto.items()
        )
        return completar(
            [{"role": "user", "content": f"Task: {paso.descripcion}\n\nAvailable context:\n{contexto_str}"}],
            system=SYSTEM_SINTETIZADOR,
            max_tokens=600
        )


def replanificar(plan: Plan, paso_fallido: Subtarea, error: str) -> list[Subtarea]:
    """
    Sugiere cómo modificar los pasos restantes tras un fallo.
    Devuelve los pasos revisados (puede saltarse el fallido o añadir alternativas).
    """
    completados = [p for p in plan.pasos if p.estado == EstadoPaso.COMPLETADO]
    pendientes  = [p for p in plan.pasos if p.estado == EstadoPaso.PENDIENTE]

    prompt = f"""A step failed. Suggest how to revise the remaining plan.

Objective: {plan.objetivo}
Failed step: {paso_fallido.descripcion}
Error: {error}

Completed steps: {[p.id for p in completados]}
Remaining steps: {[p.descripcion for p in pendientes]}

Options: skip the failed step, find an alternative approach, or simplify remaining steps.
Return JSON: {{"pasos_revisados": [{{"id": "step_X", "descripcion": "...", "herramienta": "..."|null, "argumentos": {{}}|null, "depende_de": []}}]}}"""

    try:
        datos   = completar_json([{"role": "user", "content": prompt}], system=SYSTEM_REPLANIFICADOR)
        return [
            Subtarea(
                id=p["id"] + "_r",   # sufijo _r para indicar replanificado
                descripcion=p["descripcion"],
                herramienta=p.get("herramienta"),
                argumentos=p.get("argumentos") or {},
                depende_de=p.get("depende_de", [])
            )
            for p in datos.get("pasos_revisados", [])
        ]
    except Exception:
        # Si la replanificación falla, simplemente salta el paso
        return pendientes


def ejecutar_plan(
    plan: Plan,
    max_reintentos: int = 1,
    verbose: bool = True
) -> tuple[Plan, str]:
    """
    Ejecuta el plan completo paso a paso.
    Gestiona fallos con replanificación.
    Devuelve (plan_ejecutado, respuesta_final).
    """
    plan.estado    = "en_ejecucion"
    contexto       = {}    # {id_paso: resultado}
    t_inicio_total = time.perf_counter()

    if verbose:
        print(f"\nEjecutando plan '{plan.id}'")
        print(f"Objetivo: {plan.objetivo}")
        print(f"Pasos: {len(plan.pasos)}")

    # El bucle principal: mientras haya pasos listos para ejecutar
    iteracion = 0
    while not plan.esta_completo() and iteracion < 20:
        iteracion += 1
        listos = plan.pasos_listos()
        if not listos:
            if verbose:
                print("  No hay pasos ejecutables. Posible deadlock en dependencias.")
            break

        for paso in listos:
            paso.estado = EstadoPaso.EN_CURSO
            t_paso      = time.perf_counter()

            if verbose:
                herr = f"[{paso.herramienta}]" if paso.herramienta else "[LLM]"
                print(f"\n  → {paso.id} {herr}: {paso.descripcion[:55]}")

            intento = 0
            while intento <= max_reintentos:
                try:
                    resultado    = ejecutar_paso(paso, contexto)
                    paso.resultado = resultado
                    paso.estado  = EstadoPaso.COMPLETADO
                    contexto[paso.id] = resultado

                    latencia = (time.perf_counter() - t_paso) * 1000
                    if verbose:
                        print(f"     ✓ {resultado[:90]}... ({latencia:.0f}ms)")
                    break

                except Exception as e:
                    intento += 1
                    if intento > max_reintentos:
                        paso.estado = EstadoPaso.FALLIDO
                        paso.error  = str(e)
                        if verbose:
                            print(f"     ✗ Falló: {e}")
                        # Replanifica el resto
                        nuevos_pasos = replanificar(plan, paso, str(e))
                        # Sustituye los pendientes por los replanificados
                        plan.pasos = (
                            [p for p in plan.pasos if p.estado != EstadoPaso.PENDIENTE]
                            + nuevos_pasos
                        )
                        if verbose:
                            print(f"     Replanificado: {len(nuevos_pasos)} pasos nuevos")

    # Síntesis final
    resultados_str = "\n\n".join(
        f"[{p.id}: {p.descripcion[:40]}]\n{p.resultado or 'Sin resultado.'}"
        for p in plan.pasos if p.resultado
    )

    respuesta_final = completar(
        [{"role": "user", "content": f"Objective: {plan.objetivo}\n\nResults:\n{resultados_str}\n\nProvide a clear, integrated answer."}],
        system=SYSTEM_SINTETIZADOR,
        max_tokens=800,
        temperature=0.1
    )

    plan.estado = "completado"
    latencia_total = (time.perf_counter() - t_inicio_total) * 1000

    if verbose:
        completados = sum(1 for p in plan.pasos if p.estado == EstadoPaso.COMPLETADO)
        fallidos    = sum(1 for p in plan.pasos if p.estado == EstadoPaso.FALLIDO)
        print(f"\n{'='*55}")
        print(f"Plan completado en {latencia_total:.0f}ms")
        print(f"Pasos: {completados} OK, {fallidos} fallidos")

    return plan, respuesta_final


# ── Ejecuta los planes generados en la celda anterior ─────────────────────────
plan_a_ejecutar = planes_generados[0]   # PostgreSQL vs MongoDB
plan_ejecutado, respuesta = ejecutar_plan(plan_a_ejecutar, verbose=True)

plan_ejecutado.mostrar()
print(f"\nRESPUESTA FINAL:\n{'='*55}")
print(respuesta)

---
## Celda 3 — Sistema multi-agente: Supervisor + Workers

El supervisor delega subtareas a agentes especializados.
Cada worker tiene un rol y un conjunto de herramientas diferente.

In [ ]:
@dataclass
class Delegacion:
    """Registro de una delegación del supervisor a un worker."""
    ronda:      int
    worker:     str
    subtarea:   str
    resultado:  str
    latencia_ms: float


# ── Definición de los workers ─────────────────────────────────────────────────

def worker_investigador(subtarea: str) -> str:
    """
    Worker especializado en buscar información factual.
    Usa Wikipedia como fuente principal.
    """
    system = """You are a research specialist. Your job is to find accurate factual information.
Use the buscar_wikipedia tool to find information.
Return a structured summary of what you found, citing the source."""

    # Este worker hace una búsqueda directa (simplificado: sin bucle de agente)
    resultado_wiki = buscar_wikipedia(subtarea, sentences=5)
    return completar(
        [{"role": "user", "content": f"Summarize this for: {subtarea}\n\n{resultado_wiki}"}],
        system=system, max_tokens=300
    )


def worker_analista(subtarea: str) -> str:
    """
    Worker especializado en análisis y cálculos.
    Puede hacer cálculos matemáticos y análisis comparativos.
    """
    system = """You are a data analyst. Your job is to analyze information and perform calculations.
Use the calcular tool for any mathematical operations.
Return a structured analysis with numbers and clear reasoning."""

    # Intenta extraer y ejecutar cálculos si la tarea los requiere
    return completar(
        [{"role": "user", "content": f"Analyze this: {subtarea}\n\nDate for reference: {obtener_fecha()}"}],
        system=system, max_tokens=300
    )


def worker_redactor(subtarea: str) -> str:
    """
    Worker especializado en redactar contenido estructurado.
    Toma información cruda y la convierte en texto claro y bien formateado.
    """
    system = """You are a technical writer. Your job is to produce clear, structured content.
Format your output with headers and bullet points.
Be concise, avoid repetition, and make the content actionable."""

    return completar(
        [{"role": "user", "content": subtarea}],
        system=system, max_tokens=400
    )


WORKERS: dict[str, Callable] = {
    "investigador": worker_investigador,
    "analista":     worker_analista,
    "redactor":     worker_redactor,
}

WORKERS_DESC = {
    "investigador": "Finds factual background information from reliable sources",
    "analista":     "Performs analysis, comparisons, and calculations",
    "redactor":     "Writes structured, clear content from raw information",
}


# ── El supervisor ─────────────────────────────────────────────────────────────

SYSTEM_SUPERVISOR = """You are a supervisor managing a team of specialized AI workers.
Your team: investigador (research), analista (analysis/calculations), redactor (writing).
For each round, decide: delegate to a worker OR produce the final answer.
Always end with the redactor to produce a polished final response."""


def supervisor_decide(
    objetivo: str,
    delegaciones: list[Delegacion]
) -> dict:
    """
    El supervisor decide el siguiente paso: delegar o responder.
    Devuelve {"accion": "delegar"|"responder", "worker": "...", "subtarea": "..."}
    """
    historial = [{"ronda": d.ronda, "worker": d.worker,
                  "subtarea": d.subtarea, "resultado": d.resultado[:150]}
                 for d in delegaciones]

    workers_str = json.dumps(WORKERS_DESC)
    prompt = f"""Objective: {objetivo}

Work done so far:
{json.dumps(historial, indent=2)}

Available workers: {workers_str}

What to do next?
- If more information or analysis is needed: delegate to a worker
- If we have enough to produce a final answer: respond directly

Return JSON: {{"accion": "delegar"|"responder", "worker": "investigador"|"analista"|"redactor"|null, "subtarea": "specific task for the worker"|null, "razon": "why this decision"}}"""

    return completar_json([{"role": "user", "content": prompt}], system=SYSTEM_SUPERVISOR)


def ejecutar_multiagente(
    objetivo: str,
    max_rondas: int = 5,
    verbose: bool = True
) -> tuple[list[Delegacion], str]:
    """
    Sistema multi-agente Supervisor + Workers.
    El supervisor orquesta hasta max_rondas rondas de delegación.
    """
    delegaciones: list[Delegacion] = []
    t_inicio = time.perf_counter()

    if verbose:
        print(f"\nSistema multi-agente")
        print(f"Objetivo: {objetivo}")
        print("=" * 60)

    for ronda in range(1, max_rondas + 1):
        decision = supervisor_decide(objetivo, delegaciones)

        if decision["accion"] == "responder" or decision.get("worker") is None:
            if verbose:
                print(f"\n[Ronda {ronda}] Supervisor: respuesta final")
            break

        worker_nombre = decision["worker"]
        subtarea      = decision["subtarea"]
        razon         = decision.get("razon", "")

        if verbose:
            print(f"\n[Ronda {ronda}] Supervisor → {worker_nombre}")
            print(f"  Tarea:  {subtarea[:70]}")
            print(f"  Razón:  {razon[:60]}")

        if worker_nombre not in WORKERS:
            if verbose:
                print(f"  Worker '{worker_nombre}' no existe. Saltando.")
            continue

        t_worker = time.perf_counter()
        resultado = WORKERS[worker_nombre](subtarea)
        latencia  = (time.perf_counter() - t_worker) * 1000

        delegaciones.append(Delegacion(
            ronda=ronda, worker=worker_nombre,
            subtarea=subtarea, resultado=resultado,
            latencia_ms=round(latencia, 1)
        ))

        if verbose:
            print(f"  Resultado: {resultado[:120]}... ({latencia:.0f}ms)")

    # Síntesis final por el redactor
    contexto_final = "\n\n".join(
        f"[{d.worker}, ronda {d.ronda}]\n{d.resultado}" for d in delegaciones
    )
    respuesta_final = worker_redactor(
        f"Produce a final report for this objective: {objetivo}\n\nResearch gathered:\n{contexto_final}"
    )

    latencia_total = (time.perf_counter() - t_inicio) * 1000
    if verbose:
        print(f"\nTotal: {len(delegaciones)} delegaciones en {latencia_total:.0f}ms")

    return delegaciones, respuesta_final


# ── Prueba del sistema multi-agente ───────────────────────────────────────────
OBJETIVO_MULTIAGENTE = (
    "Write a brief technology assessment: Is FastAPI a good choice for "
    "building AI-powered APIs in 2024? Include history, current state, and recommendation."
)

delegaciones, informe = ejecutar_multiagente(OBJETIVO_MULTIAGENTE, max_rondas=4, verbose=True)

print(f"\n{'='*60}")
print("INFORME FINAL")
print(f"{'='*60}")
print(informe)

**Observaciones** (edita esta celda):

- ¿El supervisor distribuyó bien las tareas entre los workers?
- ¿Cuántas rondas necesitó? ¿Crees que fueron necesarias todas?
- Compara la latencia del multi-agente con el Plan-and-Execute. ¿Cuándo usarías cada uno?

---
## Reto final — Pipeline de inteligencia competitiva

Combina Plan-and-Execute con el sistema multi-agente para producir
un informe de análisis competitivo entre dos tecnologías.
El planificador estructura el trabajo; el sistema multi-agente lo ejecuta.

Lee el esqueleto y completa los `# TODO`.

In [ ]:
@dataclass
class InformeCompetitivo:
    tecnologia_a:  str
    tecnologia_b:  str
    plan:          Plan | None
    delegaciones:  list[Delegacion]
    informe_final: str | None
    latencia_ms:   float

    def mostrar_resumen(self) -> None:
        print(f"\nInforme: {self.tecnologia_a} vs {self.tecnologia_b}")
        print(f"Pasos ejecutados: {len(self.plan.pasos) if self.plan else 0}")
        print(f"Delegaciones:     {len(self.delegaciones)}")
        print(f"Latencia total:   {self.latencia_ms:.0f}ms")
        if self.informe_final:
            print(f"\nINFORME:\n{'='*60}")
            print(self.informe_final)


def analisis_competitivo(
    tecnologia_a: str,
    tecnologia_b: str,
    verbose: bool = True
) -> InformeCompetitivo:
    """
    Pipeline completo de análisis competitivo entre dos tecnologías.

    TODO: implementa el pipeline en dos fases:

    FASE 1 — Planificación:
      Crea un plan para recopilar información sobre ambas tecnologías.
      El objetivo del plan debe incluir:
      - Historia y origen de cada tecnología
      - Casos de uso principales
      - Diferencias clave
      Usa crear_plan() con un objetivo bien formulado.

    FASE 2 — Ejecución multi-agente:
      Para cada paso del plan que implique investigación,
      delega al worker_investigador.
      Para el paso de síntesis final, delega al worker_redactor.

      SIMPLIFICACIÓN ACEPTABLE: en lugar de integrar el plan con el
      sistema multi-agente (que requiere mucho glue code), puedes:
      1. Ejecutar ejecutar_plan() para recopilar información
      2. Luego pasar el contexto completo al worker_redactor para el informe

    Tip: el objetivo del plan puede ser:
      f"Research {tecnologia_a} and {tecnologia_b}: history, use cases, strengths, weaknesses."
    """
    t_inicio = time.perf_counter()

    # TODO: implementa las dos fases
    raise NotImplementedError("Implementa analisis_competitivo()")


# ── Test del pipeline ─────────────────────────────────────────────────────────
# Descomenta cuando hayas implementado analisis_competitivo()

# COMPARACIONES = [
#     ("FastAPI",      "Django REST Framework"),
#     ("PostgreSQL",   "MongoDB"),
#     ("Docker",       "Kubernetes"),
# ]

# for tech_a, tech_b in COMPARACIONES[:1]:   # empieza con una sola comparación
#     informe = analisis_competitivo(tech_a, tech_b, verbose=True)
#     informe.mostrar_resumen()

### Extensiones opcionales

1. **Ejecución paralela real de pasos independientes:** en `ejecutar_plan()`, en lugar de ejecutar los `pasos_listos` en serie, usa `concurrent.futures.ThreadPoolExecutor` para ejecutarlos en paralelo. Mide la reducción de latencia.

2. **Worker con memoria:** da al `worker_investigador` acceso a la colección Chroma del notebook 07 (`rag_docs`). Si la información está en el corpus interno, la usa; si no, recurre a Wikipedia.

3. **Export del informe a Markdown:** después de `analisis_competitivo()`, guarda el informe en un archivo `.md` con frontmatter (fecha, tecnologías comparadas, número de fuentes). Será la base del Proyecto 2 del bloque.

In [ ]:
# Extensión 1: paralelización con ThreadPoolExecutor
# Descomenta y completa para ejecutar pasos independientes en paralelo

# from concurrent.futures import ThreadPoolExecutor, as_completed

# def ejecutar_plan_paralelo(plan: Plan, verbose: bool = True) -> tuple[Plan, str]:
#     """
#     Versión paralela de ejecutar_plan().
#     Los pasos sin dependencias entre sí se ejecutan en paralelo.
#     """
#     plan.estado = "en_ejecucion"
#     contexto    = {}
#     t_inicio    = time.perf_counter()

#     while not plan.esta_completo():
#         listos = plan.pasos_listos()
#         if not listos:
#             break

#         # TODO: ejecuta todos los pasos listos en paralelo
#         # con ThreadPoolExecutor
#         ...

#     # TODO: síntesis final
#     ...

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
## Próxima semana

La semana 12 es el **Proyecto 1 del portfolio**: un RAG chatbot completo
con agente, memoria y evaluación automática sobre un corpus real de tu dominio
(agri-food, biotech o forestal). Será tu primera pieza de portfolio para clientes.

---
*Notebook 11 — Semana 11 del Plan de Formación AI Engineering*